# LeetCode 3801 — Minimum Cost to Merge Sorted Lists bằng Python

- Link: https://leetcode.com/problems/minimum-cost-to-merge-sorted-lists/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên tập con (bitmask DP), duyệt tập con của một mask (`sub = (sub − 1) & mask`), trung vị của hợp nhiều dãy đã sắp (tìm nhị phân trên giá trị + `bisect`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minMergeCost(self, lists: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra: độ dài và trung vị của một danh sách **sau khi gộp** chỉ phụ thuộc **tập danh sách gốc** đã gộp vào nó, không phụ thuộc thứ tự gộp.
2. Mô hình hóa quá trình gộp thành **cây nhị phân** trên tập danh sách ⇒ DP `f[mask] = cnt[mask] + min_{mask = A ⊔ B} (f[A] + f[B] + |med A − med B|)`.
3. Duyệt mọi cách chia một mask thành hai phần bằng vòng lặp tập con — tổng `O(3^n)`.
4. Tính trung vị của hợp nhiều dãy đã sắp bằng **tìm nhị phân trên giá trị** + `bisect`.
5. Kiểm thử với vét cạn mọi thứ tự gộp.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Có `n ≤ 12` danh sách đã sắp. Mỗi bước chọn hai danh sách `a`, `b`, gộp thành một danh sách đã sắp với chi phí `len(a) + len(b) + |median(a) − median(b)|` (trung vị: phần tử giữa; nếu độ dài chẵn lấy phần tử **giữa bên trái**). Lặp tới khi còn một danh sách. Tìm tổng chi phí nhỏ nhất.

**Ví dụ 1:** `[[1,3,5],[2,4],[6,7,8]]` → `18` (gộp hai danh sách đầu: `5 + |3 − 2| = 6`; rồi với `[6,7,8]`: `8 + |3 − 7| = 12`).

**Ví dụ 2:** `[[1,1,5],[1,4,7,8]]` → `10`.

**Ví dụ 3:** `[[1],[3]]` → `4`. **Ví dụ 4:** `[[1],[1]]` → `2`.

**Ràng buộc:** `2 ≤ n ≤ 12`, mỗi danh sách dài `1..500`, tổng độ dài `≤ 2000`, giá trị `|x| ≤ 10^9`.

In [1]:
from typing import List
from bisect import bisect_right
from functools import cache
import random, time

INF = float('inf')
examples = [([[1, 3, 5], [2, 4], [6, 7, 8]], 18), ([[1, 1, 5], [1, 4, 7, 8]], 10), ([[1], [3]], 4), ([[1], [1]], 2)]
for L, exp in examples:
    print(L, "-> kỳ vọng", exp)

[[1, 3, 5], [2, 4], [6, 7, 8]] -> kỳ vọng 18
[[1, 1, 5], [1, 4, 7, 8]] -> kỳ vọng 10
[[1], [3]] -> kỳ vọng 4
[[1], [1]] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Trung vị "giữa bên trái"

Danh sách đã sắp độ dài `L`: trung vị là phần tử chỉ số `(L − 1) // 2` (0-indexed) — tức phần tử thứ `⌈L/2⌉` khi đếm từ 1.

In [2]:
def median(a):
    return sorted(a)[(len(a) - 1) // 2]

print(median([1, 3, 5]), median([2, 4]), median([1, 2, 3, 4, 5, 6, 7, 8]))

3 2 4


### 3.2. Bitmask và duyệt tập con

Tập danh sách ↔ `mask`. Duyệt mọi tập con khác rỗng của `mask`: `sub = mask; while sub: …; sub = (sub − 1) & mask`. Tổng số cặp `(mask, sub)` trên mọi mask là `3^n` (mỗi phần tử: ngoài mask / trong sub / trong mask nhưng ngoài sub).

In [3]:
mask = 0b1011
subs = []
sub = mask
while sub:
    subs.append(bin(sub)); sub = (sub - 1) & mask
print(subs)
print("3^12 =", 3**12)

['0b1011', '0b1010', '0b1001', '0b1000', '0b11', '0b10', '0b1']
3^12 = 531441


### 3.3. Phần tử nhỏ thứ `t` của hợp nhiều dãy đã sắp

Tìm nhị phân trên **giá trị** `v` (trong tập giá trị đã sắp): số phần tử `≤ v` = `Σ bisect_right(list_i, v)`. Giá trị nhỏ nhất có đếm `≥ t` là phần tử thứ `t`.

In [4]:
def kth_of_union(lists, t, vals):
    lo, hi = 0, len(vals) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(bisect_right(l, vals[mid]) for l in lists) >= t:
            hi = mid
        else:
            lo = mid + 1
    return vals[lo]

ls = [[1, 3, 5], [2, 4]]
vals = sorted({x for l in ls for x in l})
print([kth_of_union(ls, t, vals) for t in range(1, 6)], sorted(ls[0] + ls[1]))

[1, 2, 3, 4, 5] [1, 2, 3, 4, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

**Quan sát:** danh sách thu được sau khi gộp một tập `S` các danh sách gốc là **hợp đã sắp** của chúng — không phụ thuộc thứ tự gộp. Do đó `len(S)` và `med(S)` là **hàm của tập** `S`.

**Cây gộp:** mọi quá trình gộp tạo ra một cây nhị phân có lá là các danh sách gốc. Mỗi nút trong ứng với tập `S = A ⊔ B` (hai con), chi phí `len(A) + len(B) + |med(A) − med(B)| = len(S) + |med(A) − med(B)|`. Tổng chi phí = tổng trên mọi nút trong.

Tính tay ví dụ 1 (`[1,3,5]` = 0, `[2,4]` = 1, `[6,7,8]` = 2): ba cây có thể (chọn cặp gộp trước):

| gộp trước | chi phí bước 1 | gộp sau | chi phí bước 2 | tổng |
|---|---|---|---|---|
| {0,1} | 5 + |3−2| = 6 | với {2}: med {0,1} = 3 | 8 + |3−7| = 12 | **18** |
| {0,2} | 6 + |3−7| = 10 | med {0,2} = 5 (từ 1,3,5,6,7,8) | 8 + |5−2| = 11 | 21 |
| {1,2} | 5 + |2−7| = 10 | med {1,2} = 6 (2,4,6,7,8) | 8 + |3−6| = 11 | 21 |

## 5. Cách 1 — Brute Force: thử mọi thứ tự gộp

Đệ quy: với danh sách các danh sách hiện có, thử mọi cặp để gộp. Số trường hợp `∏ C(k, 2)` — chỉ dùng cho `n ≤ 6`.

In [5]:
def brute(lists):
    def go(cur):
        if len(cur) == 1:
            return 0
        best = INF
        for i in range(len(cur)):
            for j in range(i + 1, len(cur)):
                a, b = cur[i], cur[j]
                cost = len(a) + len(b) + abs(median(a) - median(b))
                nxt = [cur[t] for t in range(len(cur)) if t != i and t != j] + [sorted(a + b)]
                best = min(best, cost + go(nxt))
        return best
    return go([list(l) for l in lists])

for L, exp in examples:
    print(brute(L), "kỳ vọng", exp)

18 kỳ vọng 18
10 kỳ vọng 10
4 kỳ vọng 4
2 kỳ vọng 2


**Nút thắt:** nhiều thứ tự gộp dẫn tới cùng các tập con trung gian. Vì chi phí chỉ phụ thuộc tập ⇒ DP trên tập con.

## 6. Ý tưởng thuật toán / bước đột phá

**DP trên tập con:**

- `f[{i}] = 0` (một danh sách, không cần gộp).
- `f[S] = cnt[S] + min_{A ⊔ B = S, A, B ≠ ∅} ( f[A] + f[B] + |med[A] − med[B]| )`.

Đáp án `f[toàn bộ]`. Mỗi cách chia không thứ tự xét một lần (điều kiện `A < B` hoặc `A` chứa bit thấp nhất).

**Tiền tính:** `cnt[S]` (tổng độ dài) và `med[S]` = phần tử thứ `(cnt[S] + 1) // 2` của hợp. Có `2^{12} = 4096` tập.

Độ phức tạp DP `O(3^n) ≈ 5,3·10^5`.

## 7. Các cách tiếp theo

- **Cách 2:** đệ quy có nhớ trên `mask`, `med` tính bằng cách **gộp thật** (sắp xếp hợp) — dễ viết.
- **Cách 3:** DP từ dưới lên + trung vị bằng **tìm nhị phân trên giá trị** (lời giải nộp chính).
- **Cách 4:** DP từ dưới lên + trung vị tính **tăng dần**: `merged[S] = merge(merged[S \ {thấp nhất}], lists[thấp nhất])` (lưu toàn bộ danh sách đã gộp, tốn bộ nhớ hơn).

### 7.1. Cách 2 — Đệ quy có nhớ, trung vị bằng sắp xếp

In [6]:
def memo_sorted(lists):
    n = len(lists)
    full = (1 << n) - 1

    @cache
    def info(mask):
        allv = sorted(x for i in range(n) if mask >> i & 1 for x in lists[i])
        return len(allv), allv[(len(allv) - 1) // 2]

    @cache
    def f(mask):
        if mask & (mask - 1) == 0:
            return 0
        low = mask & -mask
        best = INF
        sub = (mask - 1) & mask
        while sub:
            if sub & low:                         # mỗi cách chia xét một lần
                other = mask ^ sub
                best = min(best, f(sub) + f(other) + abs(info(sub)[1] - info(other)[1]))
            sub = (sub - 1) & mask
        return best + info(mask)[0]

    return f(full)

for L, exp in examples:
    print(memo_sorted(L), "kỳ vọng", exp)

18 kỳ vọng 18
10 kỳ vọng 10
4 kỳ vọng 4
2 kỳ vọng 2


**Độ phức tạp:** `info` cho 4096 mask, mỗi lần sắp tới 2000 phần tử ⇒ ~`10^7`·log; DP `O(3^n)`.

### 7.2. Cách 3 — Từ dưới lên + trung vị bằng tìm nhị phân (lời giải chính)

In [7]:
def bottom_up(lists):
    n = len(lists)
    vals = sorted({x for l in lists for x in l})
    N = 1 << n
    cnt = [0] * N
    med = [0] * N
    for mask in range(1, N):
        low = (mask & -mask).bit_length() - 1
        cnt[mask] = cnt[mask & (mask - 1)] + len(lists[low])
        need = (cnt[mask] + 1) // 2
        members = [lists[i] for i in range(n) if mask >> i & 1]
        lo, hi = 0, len(vals) - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if sum(bisect_right(l, vals[mid]) for l in members) >= need:
                hi = mid
            else:
                lo = mid + 1
        med[mask] = vals[lo]
    f = [INF] * N
    for mask in range(1, N):
        if mask & (mask - 1) == 0:
            f[mask] = 0
            continue
        low = mask & -mask
        best = INF
        sub = (mask - 1) & mask
        while sub:
            if sub & low:
                other = mask ^ sub
                v = f[sub] + f[other] + abs(med[sub] - med[other])
                if v < best: best = v
            sub = (sub - 1) & mask
        f[mask] = best + cnt[mask]
    return f[N - 1]

for L, exp in examples:
    print(bottom_up(L), "kỳ vọng", exp)

18 kỳ vọng 18
10 kỳ vọng 10
4 kỳ vọng 4
2 kỳ vọng 2


**Độ phức tạp Cách 3:** tiền tính `O(2^n · n · log V · log L)`, DP `O(3^n)`, bộ nhớ `O(2^n)`.

### 7.3. Cách 4 — Gộp tăng dần (lưu danh sách đã gộp)

`merged[S] = gộp(merged[S bỏ phần tử thấp nhất], lists[thấp nhất])` bằng `heapq.merge` — mỗi mask tốn `O(cnt[S])`. Tổng bộ nhớ có thể tới `2^{12} · 2000 = 8·10^6` số — nặng nhưng chấp nhận được với dữ liệu nhỏ; để tiết kiệm có thể chỉ giữ theo từng tầng `popcount`.

In [8]:
import heapq
def incremental(lists):
    n = len(lists); N = 1 << n
    merged = [None] * N; merged[0] = []
    med = [0] * N; cnt = [0] * N
    for mask in range(1, N):
        low = (mask & -mask).bit_length() - 1
        merged[mask] = list(heapq.merge(merged[mask & (mask - 1)], lists[low]))
        cnt[mask] = len(merged[mask]); med[mask] = merged[mask][(cnt[mask] - 1) // 2]
    f = [0] * N
    for mask in range(1, N):
        if mask & (mask - 1) == 0: continue
        low = mask & -mask; best = INF; sub = (mask - 1) & mask
        while sub:
            if sub & low:
                other = mask ^ sub
                best = min(best, f[sub] + f[other] + abs(med[sub] - med[other]))
            sub = (sub - 1) & mask
        f[mask] = best + cnt[mask]
    return f[N - 1]

for L, exp in examples:
    print(incremental(L), "kỳ vọng", exp)

18 kỳ vọng 18
10 kỳ vọng 10
4 kỳ vọng 4
2 kỳ vọng 2


In [9]:
class SolutionIncremental:
    def minMergeCost(self, lists: List[List[int]]) -> int:
        return incremental(lists)

print([SolutionIncremental().minMergeCost(L) for L, _ in examples])

[18, 10, 4, 2]


**Độ phức tạp Cách 4:** `O(Σ_S cnt[S]) ≤ 2^n · 2000` thời gian và bộ nhớ ⇒ khoảng `8·10^6` — nhanh hơn tìm nhị phân nhưng tốn bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1: in `cnt`, `med`, `f` của mọi mask.

In [10]:
def trace(lists):
    n = len(lists); N = 1 << n
    info = {}
    for mask in range(1, N):
        allv = sorted(x for i in range(n) if mask >> i & 1 for x in lists[i])
        info[mask] = (len(allv), allv[(len(allv) - 1) // 2])
    f = {}
    for mask in range(1, N):
        c, m = info[mask]
        if mask & (mask - 1) == 0:
            f[mask] = 0; print(f"mask {mask:0{n}b}: cnt={c} med={m} f=0 (một danh sách)"); continue
        low = mask & -mask; best = INF; arg = None; sub = (mask - 1) & mask
        while sub:
            if sub & low:
                other = mask ^ sub
                v = f[sub] + f[other] + abs(info[sub][1] - info[other][1])
                if v < best: best, arg = v, (sub, other)
            sub = (sub - 1) & mask
        f[mask] = best + c
        print(f"mask {mask:0{n}b}: cnt={c} med={m} f={f[mask]} (chia {arg[0]:0{n}b} | {arg[1]:0{n}b})")
    return f[N - 1]

print("đáp án:", trace([[1, 3, 5], [2, 4], [6, 7, 8]]))

mask 001: cnt=3 med=3 f=0 (một danh sách)
mask 010: cnt=2 med=2 f=0 (một danh sách)
mask 011: cnt=5 med=3 f=6 (chia 001 | 010)
mask 100: cnt=3 med=7 f=0 (một danh sách)
mask 101: cnt=6 med=5 f=10 (chia 001 | 100)
mask 110: cnt=5 med=6 f=10 (chia 010 | 100)
mask 111: cnt=8 med=4 f=18 (chia 011 | 100)
đáp án: 18


## 9. Code nộp LeetCode

In [11]:
from bisect import bisect_right

class Solution:
    def minMergeCost(self, lists: List[List[int]]) -> int:
        n = len(lists)
        N = 1 << n
        vals = sorted({x for l in lists for x in l})
        cnt = [0] * N
        med = [0] * N
        for mask in range(1, N):
            low = (mask & -mask).bit_length() - 1
            cnt[mask] = cnt[mask & (mask - 1)] + len(lists[low])
            need = (cnt[mask] + 1) // 2
            members = [lists[i] for i in range(n) if mask >> i & 1]
            lo, hi = 0, len(vals) - 1
            while lo < hi:
                mid = (lo + hi) // 2
                c = 0
                for l in members:
                    c += bisect_right(l, vals[mid])
                if c >= need:
                    hi = mid
                else:
                    lo = mid + 1
            med[mask] = vals[lo]
        f = [0] * N
        for mask in range(1, N):
            if mask & (mask - 1) == 0:
                continue
            low = mask & -mask
            best = float('inf')
            sub = (mask - 1) & mask
            while sub:
                if sub & low:
                    other = mask ^ sub
                    v = f[sub] + f[other] + abs(med[sub] - med[other])
                    if v < best:
                        best = v
                sub = (sub - 1) & mask
            f[mask] = best + cnt[mask]
        return f[N - 1]

sol = Solution()
for L, exp in examples:
    got = sol.minMergeCost(L)
    print(got, "✔" if got == exp else "✘")

18 ✔
10 ✔
4 ✔
2 ✔


## 10. Vì sao thuật toán đúng

1. **Bất biến theo tập:** gộp hai danh sách đã sắp cho đúng danh sách đã sắp của hợp (multiset). Quy nạp ⇒ danh sách ứng với tập `S` luôn là `sorted(⋃_{i ∈ S} lists[i])` ⇒ độ dài và trung vị là hàm của `S`.
2. **Cây gộp ↔ phân hoạch đệ quy:** mọi quá trình gộp tương ứng một cây nhị phân đầy đủ có lá là các danh sách; ngược lại mọi cây như vậy thực hiện được (gộp theo thứ tự hậu thứ tự). Chi phí = tổng trên nút trong `(len(S) + |med(A) − med(B)|)`.
3. **Cấu trúc con tối ưu:** với gốc chia `S = A ⊔ B`, chi phí = `len(S) + |med A − med B| + chi phí(cây con A) + chi phí(cây con B)`, hai phần độc lập ⇒ `f[S]` như công thức.
4. **Tìm nhị phân trung vị:** hàm "số phần tử `≤ v`" không giảm theo `v`; giá trị nhỏ nhất đạt `≥ need` là phần tử thứ `need` của hợp.

## 11. Phân tích Time / Space Complexity

Đặt `n ≤ 12` danh sách, `L ≤ 2000` tổng độ dài, `V` số giá trị phân biệt.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi thứ tự | `∏_{k=2}^{n} C(k,2) · L` | `O(n L)` |
| 2. Memo + sắp xếp | `O(2^n L log L + 3^n)` | `O(2^n)` |
| 3. Từ dưới lên + nhị phân | `O(2^n n log V log L + 3^n)` | `O(2^n)` |
| 4. Gộp tăng dần | `O(2^n L + 3^n)` | `O(2^n L)` |

## 12. Bộ test

In [12]:
def run_all(lists, with_brute=True):
    res = [memo_sorted(lists), bottom_up(lists), incremental(lists), Solution().minMergeCost(lists)]
    if with_brute:
        res.append(brute(lists))
    return res

for L, exp in examples:
    assert all(v == exp for v in run_all(L))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    [[5], [5], [5]],                       # cùng giá trị: chỉ tốn độ dài
    [[-10**9], [10**9]],                   # chênh lệch lớn
    [[1, 2], [3, 4], [5, 6], [7, 8]],
    [[1] * 5, [2], [3] * 4],
]
for L in cases:
    r = run_all(L)
    assert len(set(r)) == 1, (L, r)
    print(L, "->", r[0])

[[5], [5], [5]] -> 5
[[-1000000000], [1000000000]] -> 2000000002
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 24
[[1, 1, 1, 1, 1], [2], [3, 3, 3, 3]] -> 18


In [14]:
# 12.3 phản ví dụ: tham lam kiểu Huffman "luôn gộp hai danh sách ngắn nhất" không tối ưu
def huffman_greedy(lists):
    cur = [sorted(l) for l in lists]; total = 0
    while len(cur) > 1:
        cur.sort(key=len); a, b = cur[0], cur[1]
        total += len(a) + len(b) + abs(median(a) - median(b))
        cur = cur[2:] + [sorted(a + b)]
    return total

rng = random.Random(1)
for _ in range(3000):
    L = [sorted(rng.randint(0, 30) for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(3, 5))]
    if huffman_greedy(L) != Solution().minMergeCost(L):
        print("phản ví dụ:", L, "| Huffman:", huffman_greedy(L), "| đúng:", Solution().minMergeCost(L)); break

phản ví dụ: [[8], [15], [12, 15, 20, 25]] | Huffman: 22 | đúng: 18


In [15]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3801)
for _ in range(150):
    L = [sorted(rng.randint(-20, 20) for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(2, 5))]
    r = run_all(L)
    assert len(set(r)) == 1, (L, r)
print("150 test có vét cạn: OK")
for _ in range(10):
    L = [sorted(rng.randint(-10**9, 10**9) for _ in range(rng.randint(1, 30))) for _ in range(rng.randint(6, 9))]
    r = run_all(L, with_brute=False)
    assert len(set(r)) == 1
print("10 test vừa: OK")

150 test có vét cạn: OK


10 test vừa: OK


In [16]:
# 12.5 test lớn nhất: 12 danh sách, tổng độ dài 2000
rng = random.Random(9)
sizes = [166] * 11 + [2000 - 166 * 11]
L = [sorted(rng.randint(-10**9, 10**9) for _ in range(s)) for s in sizes]
t0 = time.time(); a = Solution().minMergeCost(L); t1 = time.time()
b = incremental(L)
print("đáp án:", a, "| khớp Cách 4:", a == b, "| thời gian:", round(t1 - t0, 2), "s")

đáp án: 458741419 | khớp Cách 4: True | thời gian: 0.09 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trung vị dùng phần tử **giữa bên phải** khi độ dài chẵn

Đề định nghĩa trung vị là phần tử giữa **bên trái**: chỉ số `(L − 1) // 2`. Dùng `L // 2` sai với độ dài chẵn.

In [17]:
def wrong_median(a):
    return sorted(a)[len(a) // 2]                # SAI với độ dài chẵn

print("sai:", wrong_median([2, 4]), "| đúng:", median([2, 4]))

sai: 4 | đúng: 2


### Lỗi 2 — Quên cộng `cnt[mask]` (phần độ dài) vào `f`

In [18]:
def wrong_no_len(lists):
    n = len(lists); N = 1 << n
    med = [0] * N
    for mask in range(1, N):
        allv = sorted(x for i in range(n) if mask >> i & 1 for x in lists[i]); med[mask] = allv[(len(allv) - 1) // 2]
    f = [0] * N
    for mask in range(1, N):
        if mask & (mask - 1) == 0: continue
        low = mask & -mask; best = INF; sub = (mask - 1) & mask
        while sub:
            if sub & low: best = min(best, f[sub] + f[mask ^ sub] + abs(med[sub] - med[mask ^ sub]))
            sub = (sub - 1) & mask
        f[mask] = best                              # SAI: thiếu + cnt[mask]
    return f[N - 1]

print("sai:", wrong_no_len(examples[0][0]), "| đúng:", Solution().minMergeCost(examples[0][0]))

sai: 5 | đúng: 18


### Lỗi 3 — Coi trung vị của danh sách gộp là trung bình (hoặc trung vị của hai trung vị)

Trung vị của hợp **không** suy ra được từ trung vị của hai phần. Phải tính từ toàn bộ phần tử.

In [19]:
a, b = [1, 2, 3, 100, 101], [50]
print("trung vị của hai trung vị:", median([median(a), median(b)]), "| trung vị thật của hợp:", median(a + b))

trung vị của hai trung vị: 3 | trung vị thật của hợp: 3


### Lỗi 4 — Duyệt cả `(A, B)` và `(B, A)` — đúng nhưng gấp đôi thời gian

Không sai, nhưng tốn gấp đôi; điều kiện `sub & low` (phần chứa phần tử thấp nhất) xét mỗi cách chia đúng một lần.

In [20]:
def count_splits(n, dedupe):
    N = 1 << n; c = 0
    for mask in range(1, N):
        low = mask & -mask; sub = (mask - 1) & mask
        while sub:
            if not dedupe or sub & low: c += 1
            sub = (sub - 1) & mask
    return c

print("số lần xét với n = 12: không khử trùng", count_splits(12, False), "| khử trùng", count_splits(12, True))

số lần xét với n = 12: không khử trùng 523250 | khử trùng 261625


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **`n ≤ 12–16` đối tượng, gộp dần, chi phí phụ thuộc tập đã gộp** ⇒ DP trên tập con với chia đôi `O(3^n)`.
- **Đại lượng của "kết quả gộp" chỉ phụ thuộc tập** (tổng, độ dài, trung vị, max…) ⇒ tiền tính theo mask.
- **Phần tử thứ `t` của hợp nhiều dãy đã sắp** ⇒ tìm nhị phân trên giá trị + `bisect` (LC 4 dạng tổng quát).
- Khác với *Huffman / Minimum Cost to Merge Stones*: ở đây không có ràng buộc liền kề và chi phí có thành phần trung vị ⇒ tham lam không đúng.

## 15. Biến thể của bài

1. **Chi phí chỉ là độ dài** (`len(a) + len(b)`): bài Huffman — tham lam gộp hai danh sách ngắn nhất là tối ưu.
2. **Chỉ được gộp hai danh sách kề nhau**: DP đoạn `O(n³)` (*Minimum Cost to Merge Stones*, LC 1000 với `K = 2`).
3. **Chi phí dùng max/min** thay cho trung vị: cùng khung DP tập con.
4. **Gộp `k` danh sách cùng lúc**: chia mask thành `k` phần — phức tạp hơn.

In [21]:
# Biến thể 1: chi phí chỉ là độ dài -> Huffman tối ưu, đối chiếu với DP tập con
def huffman_len(sizes):
    h = list(sizes); heapq.heapify(h); tot = 0
    while len(h) > 1:
        a = heapq.heappop(h); b = heapq.heappop(h); tot += a + b; heapq.heappush(h, a + b)
    return tot

def dp_len(sizes):
    n = len(sizes); N = 1 << n; cnt = [0] * N; f = [0] * N
    for mask in range(1, N):
        low = (mask & -mask).bit_length() - 1; cnt[mask] = cnt[mask & (mask - 1)] + sizes[low]
        if mask & (mask - 1) == 0: continue
        lb = mask & -mask; best = INF; sub = (mask - 1) & mask
        while sub:
            if sub & lb: best = min(best, f[sub] + f[mask ^ sub])
            sub = (sub - 1) & mask
        f[mask] = best + cnt[mask]
    return f[N - 1]

rng = random.Random(15)
for _ in range(200):
    sizes = [rng.randint(1, 30) for _ in range(rng.randint(2, 8))]
    assert huffman_len(sizes) == dp_len(sizes)
print("Biến thể 1: Huffman khớp DP tập con.")

Biến thể 1: Huffman khớp DP tập con.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi thứ tự | Đệ quy trên danh sách hiện có | siêu mũ | `O(nL)` | Kiểm thử `n ≤ 6` |
| 2. Memo + sắp xếp | `@cache` theo mask | `O(2^n L log L + 3^n)` | `O(2^n)` | Viết nhanh |
| 3. Dưới lên + nhị phân | Trung vị bằng `bisect` | `O(2^n n log² + 3^n)` | `O(2^n)` | **Lời giải chuẩn**, ít bộ nhớ |
| 4. Gộp tăng dần | `heapq.merge` theo mask | `O(2^n L + 3^n)` | `O(2^n L)` | Khi bộ nhớ thoải mái |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minMergeCost([[2], [2], [2]])` = ?

**Bài 2 (tính tay).** `minMergeCost([[1], [10], [11]])` — thử ba thứ tự.

**Bài 3 (điền TODO).** Viết `median_of_union(lists, mask)` bằng tìm nhị phân trên giá trị.

**Bài 4 (tự viết lại).** Truy vết cây gộp tối ưu: trả về danh sách các bước `(tập A, tập B)` theo thứ tự thực hiện.

**Bài 5 (mini challenge).** Biến thể: chi phí `len(a) + len(b) + |max(a) − max(b)|`. Viết DP tập con và kiểm chứng với vét cạn.

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def median_of_union(lists, mask):
    # TODO: need = (tổng độ dài + 1) // 2; tìm nhị phân trên giá trị
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def max_variant(lists):
    # TODO: mx[mask] = max các phần tử; f như bài gốc với |mx[A] - mx[B]|
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mọi trung vị bằng 2. (2) Chi phí chỉ là độ dài: gộp hai cái đầu `2`, rồi `3`. (3) `5`.

**Bài 2.** (1) Gộp `{1},{10}`: `2 + 9 = 11`, rồi với `{11}`: med `{1,10}` = 1 ⇒ `3 + 10 = 13` ⇒ 24. (2) `{10},{11}` trước: `2 + 1 = 3`, med = 10, rồi `3 + 9 = 12` ⇒ 15. (3) `{1},{11}`: `2 + 10`, med 1, rồi `3 + 9` ⇒ 24. Đáp án `15`.

**Bài 3.** (1) Lấy các danh sách thuộc mask. (2) `vals` = tập giá trị đã sắp. (3) `bisect_right` để đếm.

**Bài 4.** (1) Lưu `arg[mask]` = cách chia tốt nhất. (2) Đệ quy từ mask đầy, in bước con trước. (3) Kiểm tra tổng chi phí khớp.

**Bài 5.** (1) `mx[mask] = max(mx[mask \ low], lists[low][-1])`. (2) Thay `med` bằng `mx`. (3) Vét cạn như Cách 1 với hàm chi phí mới.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().minMergeCost([[2], [2], [2]]) == 5 == brute([[2], [2], [2]])
assert Solution().minMergeCost([[1], [10], [11]]) == 15 == brute([[1], [10], [11]])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3
def median_of_union_sol(lists, mask):
    members = [l for i, l in enumerate(lists) if mask >> i & 1]
    total = sum(len(l) for l in members); need = (total + 1) // 2
    vals = sorted({x for l in members for x in l})
    return kth_of_union(members, need, vals)

rng = random.Random(3)
for _ in range(300):
    L = [sorted(rng.randint(-9, 9) for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 5))]
    mask = rng.randint(1, (1 << len(L)) - 1)
    allv = sorted(x for i, l in enumerate(L) if mask >> i & 1 for x in l)
    assert median_of_union_sol(L, mask) == allv[(len(allv) - 1) // 2]
print("Bài 3 OK")

Bài 3 OK


In [26]:
# Bài 4: truy vết cây gộp
def merge_plan(lists):
    n = len(lists); N = 1 << n
    info = {}
    for mask in range(1, N):
        allv = sorted(x for i in range(n) if mask >> i & 1 for x in lists[i]); info[mask] = (len(allv), allv[(len(allv) - 1) // 2])
    f = [0] * N; arg = [None] * N
    for mask in range(1, N):
        if mask & (mask - 1) == 0: continue
        low = mask & -mask; best = INF; sub = (mask - 1) & mask
        while sub:
            if sub & low:
                v = f[sub] + f[mask ^ sub] + abs(info[sub][1] - info[mask ^ sub][1])
                if v < best: best, arg[mask] = v, (sub, mask ^ sub)
            sub = (sub - 1) & mask
        f[mask] = best + info[mask][0]
    steps = []
    def walk(mask):
        if arg[mask] is None: return
        a, b = arg[mask]; walk(a); walk(b)
        steps.append(([i for i in range(n) if a >> i & 1], [i for i in range(n) if b >> i & 1]))
    walk(N - 1)
    return f[N - 1], steps

print(merge_plan(examples[0][0]))

(18, [([0], [1]), ([0, 1], [2])])


In [27]:
# Bài 5: chi phí theo max
def max_variant_sol(lists):
    n = len(lists); N = 1 << n; cnt = [0] * N; mx = [-INF] * N; f = [0] * N
    for mask in range(1, N):
        low = (mask & -mask).bit_length() - 1
        cnt[mask] = cnt[mask & (mask - 1)] + len(lists[low]); mx[mask] = max(mx[mask & (mask - 1)], lists[low][-1])
        if mask & (mask - 1) == 0: continue
        lb = mask & -mask; best = INF; sub = (mask - 1) & mask
        while sub:
            if sub & lb: best = min(best, f[sub] + f[mask ^ sub] + abs(mx[sub] - mx[mask ^ sub]))
            sub = (sub - 1) & mask
        f[mask] = best + cnt[mask]
    return f[N - 1]

def brute_max(lists):
    def go(cur):
        if len(cur) == 1: return 0
        best = INF
        for i in range(len(cur)):
            for j in range(i + 1, len(cur)):
                a, b = cur[i], cur[j]
                nxt = [cur[t] for t in range(len(cur)) if t not in (i, j)] + [sorted(a + b)]
                best = min(best, len(a) + len(b) + abs(max(a) - max(b)) + go(nxt))
        return best
    return go([list(l) for l in lists])

rng = random.Random(5)
for _ in range(150):
    L = [sorted(rng.randint(-9, 9) for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(2, 5))]
    assert max_variant_sol(L) == brute_max(L)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# tiền tính theo mask: cnt[mask], med[mask] (phần tử thứ (cnt+1)//2 của hợp)
f[single] = 0
for mask (tăng dần):
    best = min over sub ⊂ mask, sub chứa bit thấp nhất:
           f[sub] + f[mask^sub] + |med[sub] - med[mask^sub]|
    f[mask] = best + cnt[mask]
return f[full]
```

**Checklist:**

- [ ] Trung vị là phần tử chỉ số `(L − 1) // 2` (giữa **bên trái**)?
- [ ] Cộng `cnt[mask]` (độ dài) ở mỗi bước gộp?
- [ ] Trung vị tính từ **toàn bộ** phần tử của tập, không từ trung vị con?
- [ ] Mỗi cách chia xét một lần (`sub & low`)?
- [ ] `f` của một danh sách đơn bằng 0?